In [1]:
# !pip -q install pymupdf sentence-transformers hdbscan nltk

import os, re, json, requests
import numpy as np
import pandas as pd
from tqdm import tqdm

import fitz  # PyMuPDF
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

from sentence_transformers import SentenceTransformer
import hdbscan
from sklearn.metrics.pairwise import cosine_similarity


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\yazan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
c:\Users\yazan\OneDrive\Desktop\thesis\Gap2Idea\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_json('../data/arxiv-metadata-oai-snapshot.json', lines=True)

# Clean
df['title'] = df['title'].astype(str).str.replace('\n', ' ').str.strip()
df['abstract'] = df['abstract'].astype(str).str.replace('\n', ' ').str.strip()
df['categories_list'] = df['categories'].astype(str).str.split()

# Extract year from latest version created timestamp
df['year'] = pd.to_datetime(df['versions'].apply(lambda vs: vs[-1]['created']), errors='coerce').dt.year

# Choose a focused slice (edit these)
TARGET_CATS = {'cs.LG', 'stat.ML'}  # e.g. {'cs.CL'} for NLP
MIN_YEAR = 2021
N_PAPERS = 200  # keep small for MVP

sub = df[df['categories_list'].apply(lambda cats: any(c in TARGET_CATS for c in cats))]
sub = sub[sub['year'] >= MIN_YEAR].dropna(subset=['year'])

sub = sub.sample(min(N_PAPERS, len(sub)), random_state=42).copy()
sub = sub[['id','title','abstract','year','categories']]

print("Selected papers:", len(sub))
sub.head(3)


Selected papers: 200


,id,title,abstract,year,categories
2278293,2503.13980,Empowering LLMs in Decision Games through Algo...,Large Language Models (LLMs) have exhibited im...,2025,cs.LG
2010743,2402.13103,Multivariate Functional Linear Discriminant An...,Functional linear discriminant analysis (FLDA)...,2024,cs.LG math.ST stat.TH
2502721,2512.19725,Out-of-Distribution Detection for Continual Le...,Recent years have witnessed significant progre...,2025,cs.LG


In [21]:
import os
import requests
from tqdm import tqdm
import fitz  # PyMuPDF
from concurrent.futures import ThreadPoolExecutor, as_completed

PDF_DIR = "../data/pdfs"
TXT_DIR = "../data/texts"
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(TXT_DIR, exist_ok=True)

MAX_PAGES = 12
MIN_TEXT_CHARS = 500

# Tune these based on your machine / kaggle
DOWNLOAD_WORKERS = 16
EXTRACT_WORKERS = 8

def pdf_url(arxiv_id: str) -> str:
    return f"https://arxiv.org/pdf/{arxiv_id}.pdf"

def pdf_path(arxiv_id: str) -> str:
    return os.path.join(PDF_DIR, f"{arxiv_id.replace('/', '_')}.pdf")

def txt_path(arxiv_id: str) -> str:
    return os.path.join(TXT_DIR, f"{arxiv_id.replace('/', '_')}.txt")

def download_pdf(arxiv_id: str, session: requests.Session, timeout=30) -> str | None:
    out_path = pdf_path(arxiv_id)

    # Cache hit
    if os.path.exists(out_path) and os.path.getsize(out_path) > 10_000:
        return out_path

    try:
        r = session.get(pdf_url(arxiv_id), timeout=timeout)
        if r.status_code != 200:
            return None

        with open(out_path, "wb") as f:
            f.write(r.content)

        # sanity
        if os.path.getsize(out_path) < 10_000:
            return None

        return out_path
    except Exception:
        return None

def pdf_to_text(pdf_path: str, max_pages: int = MAX_PAGES) -> str:
    doc = fitz.open(pdf_path)
    texts = []
    for i, page in enumerate(doc):
        if i >= max_pages:
            break
        texts.append(page.get_text("text"))
    doc.close()
    return "\n".join(texts)

def extract_and_save_text(arxiv_id: str) -> str | None:
    out_txt = txt_path(arxiv_id)

    # Cache hit
    if os.path.exists(out_txt) and os.path.getsize(out_txt) > MIN_TEXT_CHARS:
        return out_txt

    p = pdf_path(arxiv_id)
    if not os.path.exists(p):
        return None

    try:
        text = pdf_to_text(p, max_pages=MAX_PAGES)
        if len(text.strip()) < MIN_TEXT_CHARS:
            return None

        with open(out_txt, "w", encoding="utf-8") as f:
            f.write(text)

        return out_txt
    except Exception:
        return None


# ----------------------------
# 1) Filter IDs to process
# ----------------------------
all_ids = sub["id"].tolist()

# Skip IDs where text already exists
todo_ids = []
for arxiv_id in all_ids:
    if os.path.exists(txt_path(arxiv_id)) and os.path.getsize(txt_path(arxiv_id)) > MIN_TEXT_CHARS:
        continue
    todo_ids.append(arxiv_id)

print(f"Total IDs: {len(all_ids)}")
print(f"Need processing (no saved text yet): {len(todo_ids)}")


# ----------------------------
# 2) Parallel PDF downloads
# ----------------------------
downloaded = {}

session = requests.Session()

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
    futures = {ex.submit(download_pdf, arxiv_id, session): arxiv_id for arxiv_id in todo_ids}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Downloading PDFs"):
        arxiv_id = futures[fut]
        p = fut.result()
        if p:
            downloaded[arxiv_id] = p

print("Downloaded PDFs:", len(downloaded))


# ----------------------------
# 3) Parallel extraction + save text
# ----------------------------
saved_texts = {}

with ThreadPoolExecutor(max_workers=EXTRACT_WORKERS) as ex:
    futures = {ex.submit(extract_and_save_text, arxiv_id): arxiv_id for arxiv_id in downloaded.keys()}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Extracting & saving texts"):
        arxiv_id = futures[fut]
        tp = fut.result()
        if tp:
            saved_texts[arxiv_id] = tp

print("Saved texts:", len(saved_texts))




Total IDs: 200
Need processing (no saved text yet): 15


Downloaded PDFs: 0


Extracting & saving texts: 0it [00:00, ?it/s]

Saved texts: 0


In [22]:
def rebuild_text_paths_from_disk(arxiv_ids, txt_dir=TXT_DIR, min_bytes=500):
    text_paths = {}
    for arxiv_id in arxiv_ids:
        p = os.path.join(txt_dir, f"{arxiv_id.replace('/', '_')}.txt")
        if os.path.exists(p) and os.path.getsize(p) > min_bytes:
            text_paths[arxiv_id] = p
    return text_paths

# After you define sub:
all_ids = sub["id"].tolist()
text_paths = rebuild_text_paths_from_disk(all_ids, TXT_DIR, min_bytes=500)

print("Found existing texts:", len(text_paths))
list(text_paths.items())[:3]
def rebuild_pdf_paths_from_disk(arxiv_ids, pdf_dir=PDF_DIR, min_bytes=10_000):
    pdf_paths = {}
    for arxiv_id in arxiv_ids:
        p = os.path.join(pdf_dir, f"{arxiv_id.replace('/', '_')}.pdf")
        if os.path.exists(p) and os.path.getsize(p) > min_bytes:
            pdf_paths[arxiv_id] = p
    return pdf_paths

all_ids = sub["id"].tolist()

pdf_paths  = rebuild_pdf_paths_from_disk(all_ids, PDF_DIR, min_bytes=10_000)
text_paths = rebuild_text_paths_from_disk(all_ids, TXT_DIR, min_bytes=500)

print("Found PDFs:", len(pdf_paths))
print("Found texts:", len(text_paths))
list(pdf_paths.items())[:3]



Found existing texts: 185
Found PDFs: 185
Found texts: 185


[('2512.19725', '../data/pdfs\\2512.19725.pdf'),
 ('2503.17793', '../data/pdfs\\2503.17793.pdf'),
 ('2601.03085', '../data/pdfs\\2601.03085.pdf')]

In [5]:
JSONL_TEXTS = "/kaggle/working/arxiv_texts.jsonl"
with open(JSONL_TEXTS, "w", encoding="utf-8") as f:
    for arxiv_id, txt_path in text_paths.items():
        with open(txt_path, "r", encoding="utf-8") as tf:
            text = tf.read()
        f.write(json.dumps({"id": arxiv_id, "text": text}) + "\n")
print("Wrote:", JSONL_TEXTS)


Wrote: /kaggle/working/arxiv_texts.jsonl


In [ ]:
import os, re
import fitz  # PyMuPDF
from tqdm import tqdm
import pandas as pd

# --- Config ---
MAX_TAIL_PAGES = 6          # only parse last pages for speed
MIN_SECTION_CHARS = 200     # ignore tiny sections
TARGET_SECTION_COUNT = 2    # stop after finding 2 sections
FALLBACK_MAX_WORDS = 900    # window size for fallback section-like extraction

REF_RE = re.compile(r"^\s*(references|bibliography)\s*$", re.IGNORECASE)

# Common heading forms: "4 Limitations", "IV. Future Work", "Conclusion", "DISCUSSION"
# Also allows "&" and slightly longer titles.
HEADING_RE = re.compile(
    r"^\s*(?:\d+(\.\d+)*\s+|[IVXLC]+\.\s+)?([A-Z][A-Za-z0-9 &\-/,:]{2,80})\s*$"
)

# Headings we want (and some common variants)
LIMITATION_HEAD_RE = re.compile(r"\b(limitations?|threats to validity|caveats?)\b", re.IGNORECASE)
FUTURE_HEAD_RE = re.compile(r"\b(future work|future directions?|outlook|next steps?)\b", re.IGNORECASE)

# Better fallback: find a likely section start phrase, then take a window (section-like chunk)
FALLBACK_START_RE = re.compile(
    r"(conclusion(s)?\s*(and|&)\s*future work|future work|future directions|limitations?|threats to validity|discussion|conclusion(s)?)",
    re.IGNORECASE
)

def extract_tail_text_from_pdf(pdf_path: str, tail_pages: int = MAX_TAIL_PAGES) -> str:
    """Extract text from the last N pages of a PDF for speed."""
    doc = fitz.open(pdf_path)
    n = len(doc)
    start = max(0, n - tail_pages)
    chunks = []
    for i in range(start, n):
        chunks.append(doc[i].get_text("text"))
    doc.close()
    return "\n".join(chunks)

def cut_before_references(text: str) -> str:
    """Cut text at the first 'References' heading line (if present)."""
    lines = text.splitlines()
    for i, line in enumerate(lines):
        if REF_RE.match(line.strip()):
            return "\n".join(lines[:i])
    return text  # if not found, use all tail text

def find_headings(lines):
    """Return list of (idx, heading_text) for lines that look like headings."""
    headings = []
    for i, line in enumerate(lines):
        m = HEADING_RE.match(line)
        if not m:
            continue
        title = m.group(2).strip()

        # filter obvious non-headings
        if len(title) < 3:
            continue

        # avoid lines that are mostly lowercase sentence-like
        if sum(c.islower() for c in title) > sum(c.isupper() for c in title) and " " in title:
            # allow common titles though
            if title.lower() not in {"conclusion", "conclusions", "discussion", "future work", "limitations"}:
                continue

        headings.append((i, title))
    return headings

def section_spans_from_headings(lines, headings):
    """Convert headings list to spans: heading -> (start_line, end_line)."""
    spans = []
    for k, (idx, title) in enumerate(headings):
        start = idx + 1
        end = headings[k+1][0] if k+1 < len(headings) else len(lines)
        spans.append((title, start, end))
    return spans

def extract_target_sections_from_end(text: str, target_count: int = TARGET_SECTION_COUNT):
    """
    Work backwards from end to find last N sections whose headings indicate
    Limitations/Future Work. Return list of dicts.
    """
    text = cut_before_references(text)
    lines = [ln.rstrip() for ln in text.splitlines() if ln.strip()]
    if not lines:
        return []

    headings = find_headings(lines)
    if not headings:
        return []

    spans = section_spans_from_headings(lines, headings)

    found = []
    # walk from the end (last sections first)
    for title, start, end in reversed(spans):
        body = "\n".join(lines[start:end]).strip()
        if len(body) < MIN_SECTION_CHARS:
            continue

        if LIMITATION_HEAD_RE.search(title):
            found.append({"section_type": "limitations", "heading": title, "text": body})
        elif FUTURE_HEAD_RE.search(title):
            found.append({"section_type": "future_work", "heading": title, "text": body})

        if len(found) >= target_count:
            break

    return list(reversed(found))  # keep logical order

def fallback_extract_window(text: str, max_words: int = FALLBACK_MAX_WORDS) -> str | None:
    """
    Fallback: take a coherent chunk starting at the first likely appearance of
    a limitations/future-work/conclusion/discussion start phrase.
    """
    t = cut_before_references(text)
    m = FALLBACK_START_RE.search(t)
    if not m:
        return None

    window = t[m.start():].strip()
    words = window.split()
    if len(words) > max_words:
        window = " ".join(words[:max_words])
    return window.strip()

def extract_limitations_futurework(pdf_path: str):
    """
    Main entry:
    1) tail pages
    2) cut before references
    3) extract up to 2 target sections from end
    4) fallback window if none found
    """
    tail = extract_tail_text_from_pdf(pdf_path, tail_pages=MAX_TAIL_PAGES)
    sections = extract_target_sections_from_end(tail, target_count=TARGET_SECTION_COUNT)

    if sections:
        return {"sections": sections, "fallback_text": ""}

    fb = fallback_extract_window(tail, max_words=FALLBACK_MAX_WORDS)
    return {"sections": [], "fallback_text": fb or ""}


# ---- Run over PDFs ----
rows = []

for arxiv_id, pdf_path in tqdm(pdf_paths.items()):
    out = extract_limitations_futurework(pdf_path)

    # Store found sections
    for s in out["sections"]:
        rows.append({
            "id": arxiv_id,
            "section_type": s["section_type"],
            "heading": s["heading"],
            "section_text": s["text"]
        })

    # If no sections found, store fallback window (section-like chunk)
    if not out["sections"] and out["fallback_text"]:
        rows.append({
            "id": arxiv_id,
            "section_type": "fallback",
            "heading": "fallback_window",
            "section_text": out["fallback_text"]
        })

sec_df = pd.DataFrame(rows)
print(sec_df.shape)
sec_df.head(20)


 91%|█████████▏| 169/185 [00:07<00:00, 37.47it/s]

MuPDF error: format error: object is not a stream



100%|██████████| 185/185 [00:07<00:00, 24.91it/s]

(127, 4)


,id,section_type,heading,section_text
0,2512.19725,fallback,fallback_window,"Conclusion\nIn this work, we studied the integ..."
1,2503.17793,fallback,fallback_window,Conclusion & Future Work\nWe introduce Ling-Co...
2,2601.03085,future_work,CONCLUSIONS AND FUTURE DIRECTIONS,The growing popularity of IIoT systems present...
3,2102.03018,fallback,fallback_window,Conclusion\nThe encouragement towards training...
4,2401.17544,fallback,fallback_window,discussions. 2.1 Integer Quantization Earlier ...
5,2207.09511,fallback,fallback_window,"discussion on spectral bias.\nFinally, we note..."
6,2508.13182,fallback,fallback_window,limitations. Sinoara et al. (2019) generate se...
7,2508.16261,fallback,fallback_window,Discussions & Future Directions After reviewin...
8,2305.14926,fallback,fallback_window,discussion which would help you understand the...
9,2308.13569,fallback,fallback_window,"Conclusion and Future Work\nIn conclusion, our..."


In [81]:
import json
from pathlib import Path

out_jsonl = Path("../data/sections_extracted.jsonl")
out_jsonl.parent.mkdir(parents=True, exist_ok=True)

# sec_df must have: id, section_type, heading, section_text
with out_jsonl.open("w", encoding="utf-8") as f:
    for r in sec_df[["id","section_type","heading","section_text"]].to_dict("records"):
        # ensure strings + keep newlines safely
        r["id"] = str(r["id"])
        r["section_type"] = "" if pd.isna(r["section_type"]) else str(r["section_type"])
        r["heading"] = "" if pd.isna(r["heading"]) else str(r["heading"])
        r["section_text"] = "" if pd.isna(r["section_text"]) else str(r["section_text"])
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved:", out_jsonl)


Saved: ..\data\sections_extracted.jsonl
